In [3]:
# Loaded variable 'df' from URI: /var/tmp/ge74wan/repos/supplement-kg/datasets/iDISK2.0/Data/Entity/DSP.csv
import pandas as pd
df = pd.read_csv(r'/var/tmp/ge74wan/repos/supplement-kg/datasets/iDISK2.0/Data/Entity/DSP.csv')

/tmp/ipykernel_87776/1806554044.py:3: DtypeWarning: Columns (0: NHP_ID, 1: Purpose, 2: Risk) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r'/var/tmp/ge74wan/repos/supplement-kg/datasets/iDISK2.0/Data/Entity/DSP.csv')


In [4]:
df_dsld = df[df['DSLD_ID'].notnull()]
print(df_dsld)

         iDISK_ID                                Name  \
417     DSP000417                    Heat Accelerated   
641     DSP000641                             Moringa   
1019    DSP001019                         Protein 100   
1052    DSP001052                         Resveratrol   
1791    DSP001791                    Calcifood Powder   
...           ...                                 ...   
163801  DSP163801                   Turmeric Curcumin   
163802  DSP163802   Essential for Women Mint Essenced   
163803  DSP163803          Calcium D-Glucarate 500 mg   
163804  DSP163804  Adult Probiotic Gummies Strawberry   
163805  DSP163805             Turmeric Complex 500 mg   

                      Company_Name  \
417     Magnum Nutraceuticals Inc.   
641             Genesis Today Inc.   
1019            Genesis Today Inc.   
1052            Genesis Today Inc.   
1791         Standard Process Inc.   
...                            ...   
163801              Pure By Nature   
163802     

In [5]:
df_dsld_no_nhp = df_dsld[df_dsld['NHP_ID'].isnull()]
print(df_dsld_no_nhp)

         iDISK_ID                                               Name  \
71364   DSP071364                                         B-2 100 mg   
71365   DSP071365                                         B-6 100 mg   
71366   DSP071366  C-1000 mg With Protective Bioflavonoids And Wi...   
71367   DSP071367                                      Niacin 250 mg   
71368   DSP071368                                 Niacinamide 500 mg   
...           ...                                                ...   
163801  DSP163801                                  Turmeric Curcumin   
163802  DSP163802                  Essential for Women Mint Essenced   
163803  DSP163803                         Calcium D-Glucarate 500 mg   
163804  DSP163804                 Adult Probiotic Gummies Strawberry   
163805  DSP163805                            Turmeric Complex 500 mg   

              Company_Name                                    Company_Address  \
71364   Vitamin World Inc.                   None, Ron

In [6]:
# Find products in df_dsld_no_nhp that share the same name
name_col = next(c for c in df_dsld_no_nhp.columns if 'name' in c.lower())
name_counts = df_dsld_no_nhp[name_col].value_counts()
duplicates = name_counts[name_counts > 1]

print(f'Using column: {name_col}')
print(f'Number of duplicated product names: {len(duplicates)}')
print(duplicates)

Using column: Name
Number of duplicated product names: 10747
Name
Melatonin 3 mg        95
Vitamin D3 5000 IU    74
L-Lysine 500 mg       71
Vitamin D3 1000 IU    64
Vitamin D3 2000 IU    60
                      ..
Organic Peppermint     2
Organic Spearmint      2
Chaga Powder           2
Lion's Mane Powder     2
Keto Prime             2
Name: count, Length: 10747, dtype: int64


In [7]:
df_Melatonin = df_dsld_no_nhp[df_dsld_no_nhp[name_col].str.contains('Melatonin', case=False, na=False)]
print(df_Melatonin)

         iDISK_ID                                               Name  \
71485   DSP071485                                        Melatonin 3   
71491   DSP071491                          Melatonin 1 Cherry Flavor   
71807   DSP071807                                        Melatonin 5   
71820   DSP071820              Melatonin 5000 mcg Zipmelt Peppermint   
71957   DSP071957                                     Melatonin 1 mg   
...           ...                                                ...   
163645  DSP163645  Pure Zzzs Kidz Melatonin + Chamomile & Lavende...   
163681  DSP163681          Ultra Melatonin 5 mg Natural Berry Flavor   
163684  DSP163684  Children's Sleep with Melatonin Natural Grape ...   
163732  DSP163732                                     Melatonin 1 mg   
163733  DSP163733               Melatonin 1 mg Natural Orange Flavor   

                         Company_Name  \
71485   General Nutrition Corporation   
71491   General Nutrition Corporation   
71807   Gene

## Retrieving label data from DSLD

The `DSLD_ID` column is a **pipe-delimited list**, usually with a leading empty
field (`" | 34066 | 73755"`), so one iDISK product can map to several DSLD
labels. `parse_dsld_ids` handles the splitting.

Labels come from the NIH ODS Label API at `api.ods.od.nih.gov/dsld/v9` — note
this is *not* `dsld.od.nih.gov`, which sits behind a Cloudflare interstitial
and rejects programmatic requests. No API key is needed.

One gotcha the client absorbs: an ID that is not in DSLD comes back as HTTP 200
with an empty body `[]` rather than a 404, so it is raised as `DSLDNotFound`.

In [8]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from engine.services.sources import dsld

client = dsld.DSLDClient()

# One label by its DSLD ID.
label = client.fetch(236726)

print(f'{label.dsld_id}  {label.full_name}  ({label.brand_name})')
print(f'  type       : {label.product_type} / {label.physical_state}')
print(f'  serving    : {", ".join(label.serving_sizes)} x {label.servings_per_container}')
print(f'  claims     : {label.claims}')
print(f'  target     : {label.target_groups}')
print(f'  off market : {label.off_market}   entry: {label.entry_date}')
print(f'  {len(label.ingredients)} ingredients:')
for ing in label.ingredients[:8]:
    qty = f'{ing.quantity} {ing.unit}' if ing.quantity is not None else '-'
    dv = f' ({ing.daily_value_percent}% DV)' if ing.daily_value_percent else ''
    print(f'    {ing.name:<32} {qty}{dv}  [{ing.category}]')

236726  Heat Accelerated  (Magnum Nutraceuticals)
  type       : Other Combinations / Capsule
  serving    : 1 Capsule(s) x 40
  claims     : ['All Other']
  target     : ['Adult (18 - 50 Years)', 'Women (not pregnant or lactating)', 'Gluten Free']
  off market : True   entry: 2020-09-25
  27 ingredients:
    Niacin                           4 mg (20% DV)  [vitamin]
    Guarana                          642 mg  [botanical]
    Bitter Orange                    330 mg  [botanical]
    Green Tea (Camelia sinensis) 4:1 extract 270 mg  [botanical]
    Caffeine Anhydrous               270 mg  [non-nutrient/non-botanical]
    Kola Nut (Cola nitida) 3:1 extract 270 mg  [botanical]
    Raspberry Ketone                 240 mg  [non-nutrient/non-botanical]
    Cayenne                          240 mg  [botanical]


In [9]:
# Splitting the pipe-delimited DSLD_ID column.
for cell in df_dsld['DSLD_ID'].head(5):
    print(f'{cell!r:<32} -> {dsld.parse_dsld_ids(cell)}')

# How many labels does the whole DSP table actually point at?
id_lists = df_dsld['DSLD_ID'].map(dsld.parse_dsld_ids)
print(f'\nrows with a DSLD ID : {len(df_dsld):,}')
print(f'DSLD ID references  : {id_lists.map(len).sum():,}')
print(f'unique DSLD IDs     : {len(set().union(*id_lists)):,}')
print(f'rows with >1 ID     : {(id_lists.map(len) > 1).sum():,}')

' | 236726'                      -> ['236726']
' | 34066 | 73755'               -> ['34066', '73755']
' | 35027'                       -> ['35027']
' | 34969 | 34973 | 73881'       -> ['34969', '34973', '73881']
' | 256411'                      -> ['256411']

rows with a DSLD ID : 92,650
DSLD ID references  : 156,957
unique DSLD IDs     : 156,957
rows with >1 ID     : 31,118


In [10]:
# Batch fetch, anchored to the iDISK product rows so each label keeps the
# iDISK_ID that asked for it. `limit` caps total requests -- drop it for a
# full sweep, but note that is ~117k requests at ~0.2 s each.
rows = df_Melatonin[['iDISK_ID', 'DSLD_ID']].itertuples(index=False, name=None)
labels, manifest = dsld.fetch_products(rows, client=client, limit=25)

print(f"requested {manifest['requested']}, fetched {manifest['fetched']}, "
      f"missing {len(manifest['missing'])}")

import pandas as pd
labels_df = pd.DataFrame([
    {
        'iDISK_ID': lb.idisk_id,
        'DSLD_ID': lb.dsld_id,
        'name': lb.full_name,
        'brand': lb.brand_name,
        'form': lb.physical_state,
        'serving': '; '.join(lb.serving_sizes),
        'n_ingredients': len(lb.ingredients),
        'off_market': lb.off_market,
    }
    for lb in labels
])
labels_df

requested 25, fetched 25, missing 0


,iDISK_ID,DSLD_ID,name,brand,form,serving,n_ingredients,off_market
0,DSP071485,667,Melatonin 3,GNC,Tablet or Pill,1 Tablet(s),1,True
1,DSP071485,1430,Melatonin 3,GNC,Tablet or Pill,1 Tablet(s),2,True
2,DSP071485,1431,Melatonin 3 mg,GNC,Tablet or Pill,1 Tablet(s),2,True
3,DSP071491,674,Melatonin 1 Cherry Flavor,GNC,Tablet or Pill,1 Tablet(s),1,True
4,DSP071491,675,Melatonin 1 Cherry Flavor,GNC,Tablet or Pill,1 Tablet(s),1,True
5,DSP071807,1029,Melatonin 5,GNC,Tablet or Pill,1 Tablet(s),1,True
6,DSP071820,1045,Melatonin 5000 Zipmelt 5000 mcg Peppermint,GNC,Tablet or Pill,1 Tablet(s),1,True
7,DSP071957,1202,Melatonin 1 mg,Vitamin World,Tablet or Pill,1-4 Tablet(s),1,False
8,DSP072027,1273,Melatonin 1 mg,Natrol,Tablet or Pill,1 Tablet(s),3,True
9,DSP072027,77424,Melatonin 1 mg,Natrol,Tablet or Pill,1 Tablet(s),3,True


In [ ]:
# Persist the run. Same convention as the PubMed experiment: each run gets its
# own immutable timestamped directory, payload under raw/ (gitignored) and the
# manifest under manifests/ (committable), so two runs stay comparable.
from engine.services.sources.pubmed import new_run_dir

run_dir = new_run_dir(REPO_ROOT / 'data' / 'dsld' / 'runs', label='melatonin-sample')
paths = dsld.save_labels(labels, manifest, run_dir, run_name='melatonin')

print(run_dir)
for kind, p in paths.items():
    print(f'  {kind:<9} {p.relative_to(REPO_ROOT)}')

## Why are Purpose and Risk empty for DSLD products?

**Because iDISK never extracted them from DSLD.** Purpose and Risk come from
LNHPD only — the Canadian Licensed Natural Health Products Database — so every
DSLD-only row is empty *by construction*, not by accident.

The iDISK 2.0 paper ([Hou et al., JMIR 2025](https://pmc.ncbi.nlm.nih.gov/articles/PMC11966073/))
states the extraction scope per source:

> "From **LNHPD**, we extracted the dietary supplement product (DSP) name,
> **purpose**, **safety information**, and company address information"

> "From **DSLD**, we extracted the DSP name, company address information, DSI
> name, and the relationship between DSP and DSI."

Two independent checks confirm this is the whole explanation:

1. The split is perfectly clean — **0 of 92,442** DSLD-only rows have a Purpose,
   while **91%** of NHP-sourced rows do (next cell).
2. Row counts match the paper's Table 1 exactly: 65,097 purposes and 63,623
   risks, which are precisely the non-null counts in this CSV.

### Why LNHPD has these fields and DSLD does not — the regulatory reason

The paper reports *what* was extracted but not *why* the sources differ. That
difference is regulatory, and it explains why this gap cannot be closed by
simply re-scraping DSLD harder:

- **Canada (LNHPD)** licenses natural health products **pre-market**: "All
  natural health products (NHPs) sold in Canada require a product licence
  before being marketed" ([Health Canada](https://www.canada.ca/en/health-canada/services/drugs-health-products/natural-non-prescription/applications-submissions/product-licensing.html)).
  The application specifies an authorised **recommended use or purpose**, and
  the licence issues an NPN. Purpose and risk are therefore discrete, curated,
  pre-approved database fields — iDISK can read them straight off.
- **US (DSLD)** is a **label archive**. Under DSHEA, "FDA is not authorized to
  approve dietary supplements for safety and effectiveness before they are
  marketed," and structure/function claims "are not subject to premarket review
  and authorization by FDA" ([FDA](https://www.fda.gov/food/nutrition-food-labeling-and-critical-foods/label-claims-conventional-foods-and-dietary-supplements)).
  There is no authorised-indication field to publish, so DSLD transcribes what
  is printed on the package — and benefit claims live in free text.

So the information usually **does exist** in DSLD, but unlabelled. DSLD's
`statements` vocabulary has no "purpose" type: a benefit claim typically lands
in the generic `General Statements: All Other Content` bucket, mixed in with
"Made in the USA" and "ACTUAL SIZE". Risk fares better — there are real
`Precautions re: ...` types — but it is split across several of them.

**This is exactly the gap an LLM can close**: turn DSLD's free-text label
statements into the same structured Purpose/Risk fields LNHPD supplies, and
cover the 92,442 products iDISK currently leaves blank.

⚠️ One caveat worth carrying downstream: an LNHPD purpose is a *regulator-
approved indication*, while a DSLD-derived one is an *unreviewed manufacturer
claim*. They are not the same evidentiary object, so the extracted rows are
marked with their own provenance rather than being merged into the same column
as LNHPD's.

In [ ]:
# Evidence 1: Purpose/Risk coverage splits perfectly along the source boundary.
has_nhp, has_dsld = df['NHP_ID'].notnull(), df['DSLD_ID'].notnull()
has_purpose, has_risk = df['Purpose'].notnull(), df['Risk'].notnull()

print(f"{'group':<20}{'rows':>9}{'Purpose':>10}{'Risk':>9}{'%P':>8}")
for name, mask in [
    ('NHP only',      has_nhp & ~has_dsld),
    ('DSLD only',    ~has_nhp &  has_dsld),
    ('both',          has_nhp &  has_dsld),
    ('neither',      ~has_nhp & ~has_dsld),
]:
    n = mask.sum()
    p, r = (mask & has_purpose).sum(), (mask & has_risk).sum()
    print(f'{name:<20}{n:>9,}{p:>10,}{r:>9,}{(100*p/n if n else 0):>7.1f}%')

# Evidence 2: totals reproduce the paper's Table 1 counts exactly.
print(f"\nPurpose non-null: {has_purpose.sum():,}   (paper Table 1, LNHPD: 65,097)")
print(f"Risk    non-null: {has_risk.sum():,}   (paper Table 1, LNHPD: 63,623)")

## Recovering Purpose and Risk with an LLM

`engine/services/extraction/label_purpose_risk.py` reads the retrieved label
text and returns the two missing fields as structured JSON.

Design choices that matter for a knowledge base:

- **Grounding over recall.** The prompt forbids the model from using its own
  pharmacological knowledge: if a melatonin label never states a purpose, the
  answer is `null`, not "promotes sleep". A hallucinated indication is far more
  damaging to a KG than a missing one.
- **Evidence alongside every field.** Each extraction carries the verbatim label
  text it came from, so a wrong answer is auditable without re-querying — and an
  ungrounded one is detectable by checking the evidence appears in the label.
- **Structured outputs** (`output_config.format` with a strict JSON schema), so
  responses always parse and `null` is a first-class answer.
- **`effort: "low"`** — this is short-text extraction, not reasoning, and it
  runs per-product across a large corpus.

Requires `ANTHROPIC_API_KEY` in `.env` (see `.env.example`).

In [ ]:
from engine.services.extraction import label_purpose_risk as lpr

# What the model is actually shown: label statements, filtered of pure
# boilerplate (lot codes, seals, the FDA disclaimer) but NOT pre-labelled --
# note the purpose sits in a generic "General Statements" block.
demo = client.fetch(1029)   # GNC Melatonin 5
print(lpr.label_text(demo))

In [ ]:
# Extract over the labels fetched above. Needs ANTHROPIC_API_KEY in .env.
extractions, ex_manifest = lpr.extract_many(labels)

print(f"{ex_manifest['n_extracted']}/{ex_manifest['n_labels']} extracted  |  "
      f"purpose found: {ex_manifest['n_with_purpose']}  |  "
      f"risk found: {ex_manifest['n_with_risk']}")
if ex_manifest['errors']:
    print('errors:', ex_manifest['errors'][:3])

for e in extractions[:5]:
    print(f'\n--- {e.dsld_id}  {e.name}')
    print(f'  PURPOSE : {e.purpose}')
    print(f'      from: {(e.purpose_evidence or "")[:100]}')
    print(f'  RISK    : {(e.risk or "")[:160]}')

In [ ]:
# Grounding check: is each cited evidence string actually present in the label?
# This is the cheap automatic guard against a fabricated indication -- it will
# not catch a misreading, but it does catch text the label never contained.
import re

def _norm(s):
    return re.sub(r'\s+', ' ', (s or '')).strip().lower()

by_id = {lb.dsld_id: _norm(lpr.label_text(lb)) for lb in labels}

ungrounded = []
for e in extractions:
    source = by_id.get(e.dsld_id, '')
    for fieldname in ('purpose_evidence', 'risk_evidence'):
        ev = _norm(getattr(e, fieldname))
        if ev and ev not in source:
            ungrounded.append((e.dsld_id, fieldname, ev[:80]))

checked = sum(bool(getattr(e, f)) for e in extractions
              for f in ('purpose_evidence', 'risk_evidence'))
print(f'evidence strings checked : {checked}')
print(f'not found in label text  : {len(ungrounded)}')
for row in ungrounded[:5]:
    print('  ', row)

In [ ]:
# Save extractions into the same run directory as the labels they came from,
# so the fetch and the extraction that reads it stay together.
lpr.save_extractions(extractions, ex_manifest, run_dir, run_name='purpose_risk')

# The filled-in view: iDISK's empty columns, recovered from DSLD label text.
# Provenance is explicit -- these are unreviewed manufacturer claims, NOT the
# regulator-approved indications LNHPD supplies (see the caveat above).
recovered = pd.DataFrame([
    {
        'iDISK_ID': e.idisk_id,
        'DSLD_ID': e.dsld_id,
        'Name': e.name,
        'Purpose': e.purpose,
        'Risk': e.risk,
        'purpose_source': 'dsld-label-llm' if e.purpose else None,
    }
    for e in extractions
])
print(f"purpose recovered for {recovered['Purpose'].notnull().sum()}/{len(recovered)} products "
      f"that iDISK leaves blank")
recovered